In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

In [11]:
np.random.seed(2025)

In [15]:
supply = pd.read_csv("global_supply_chain_risk_2026 (1).csv")
supply = supply.drop(columns=["Shipment_ID", "Date", "Disruption_Occurred"])
supply.head()

,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days
0,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39
1,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92
2,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54
3,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13
4,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50


In [16]:
train, test = train_test_split(supply, test_size=0.3, random_state=2025)
print("Training set dimensions:", train.shape)
print("Testing set dimensions:", test.shape)

Training set dimensions: (3500, 11)
Testing set dimensions: (1500, 11)


In [25]:
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    else:
        return ''

def print_summary_with_stars(model):
    # Get the full summary object
    summary = model.summary2()

    # Get coefficients table
    coef_table = summary.tables[1].copy()

    # Add significance stars
    coef_table['Signif'] = coef_table['P>|t|'].apply(significance_stars)

    # Round numeric columns
    numeric_columns = coef_table.select_dtypes(include=['float64', 'int64']).columns
    coef_table[numeric_columns] = coef_table[numeric_columns].round(4)

    # Print model statistics
    print("=" * 80)
    print("Regression Results")
    print("=" * 80)
    print(f"R-squared: {model.rsquared:.4f}")
    print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
    print(f"F-statistic: {model.fvalue:.4f}")
    print(f"Prob (F-statistic): {model.f_pvalue:.4e}")
    print(f"Number of observations: {int(model.nobs)}")
    print("=" * 80)

    # Print coefficients table
    print(coef_table)
    print("=" * 80)
    print("Significance codes:  '***' 0.001  '**' 0.01  '*' 0.05  '.' 0.1")
    print("=" * 80)

In [26]:
y_train = pd.to_numeric(train["Lead_Time_Days"], errors="coerce")

X_train = train.drop(columns=["Lead_Time_Days"]).copy()

for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors="ignore")

X_train = pd.get_dummies(X_train, drop_first=True)
X_train = X_train.apply(pd.to_numeric, errors="coerce")

df_model = pd.concat([y_train, X_train], axis=1).dropna()
y_train = df_model["Lead_Time_Days"]
X_train = df_model.drop(columns=["Lead_Time_Days"])

X_train = sm.add_constant(X_train).astype(float)

model = sm.OLS(y_train, X_train).fit()

print_summary_with_stars(model)

Regression Results
R-squared: 0.6310
Adjusted R-squared: 0.6277
F-statistic: 191.2787
Prob (F-statistic): 0.0000e+00
Number of observations: 3500
                                    Coef.  Std.Err.        t   P>|t|   [0.025  \
const                            -38.0349    2.9452 -12.9143  0.0000 -43.8093   
Distance_km                        0.0025    0.0001  31.9233  0.0000   0.0024   
Weight_MT                         -0.0028    0.0024  -1.1660  0.2437  -0.0074   
Fuel_Price_Index                  -0.1743    0.3520  -0.4953  0.6204  -0.8644   
Geopolitical_Risk_Score            0.9939    0.1168   8.5092  0.0000   0.7649   
Carrier_Reliability_Score          1.1997    2.3260   0.5158  0.6060  -3.3607   
Origin_Port_Busan                  0.2421    1.3371   0.1811  0.8563  -2.3795   
Origin_Port_Dubai                 -0.1523    1.3540  -0.1125  0.9105  -2.8070   
Origin_Port_Hamburg               -0.2327    1.3558  -0.1717  0.8637  -2.8910   
Origin_Port_Los Angeles            0.6080   

/tmp/ipykernel_1071/4257715949.py:6: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  X_train[col] = pd.to_numeric(X_train[col], errors="ignore")


In [20]:
y_test = test["Lead_Time_Days"]
X_test = test.drop(columns=["Lead_Time_Days"]).copy()

X_test = pd.get_dummies(X_test, drop_first=True)
X_test = X_test.astype(float)   # important line
X_test = sm.add_constant(X_test)

X_test = X_test.reindex(columns=model.model.exog_names, fill_value=0)

yhat = model.predict(X_test)

sse = np.sum((y_test - yhat)**2)
sst = np.sum((y_test - y_test.mean())**2)
osr2 = 1 - sse/sst

print("OSR^2 for model:", round(osr2, 4))

OSR^2 for model: 0.6006
